<a href="https://colab.research.google.com/github/EDUART-Bloem/114-2-Programming-Language/blob/main/HW4_HOYOLAB_GoogleSheet_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：HoYoLAB 活動 → Google Sheet → RAG（通用版）

這份 notebook 旨在演示從指定網站（目前專注於 HoYoLAB 活動頁面）爬取資料的完整流程：

1.  爬取 HoYoLAB 活動資料 (透過 API)
2.  寫入指定 Google Sheet
3.  從 Google Sheet 讀回資料
4.  建立 FAISS RAG 索引
5.  用 Gemini 根據活動資料回答問題

主要修正：專注於 HoYoLAB API 爬取，移除了 PTT 和 Playwright 相關內容，並統一使用 `gc.open_by_url(SHEET_URL)` 連接 Google Sheet。

In [4]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


In [5]:
import re
import time
import uuid
import json
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。
`HOYOLAB_WORKSHEET_NAME` 是存放 HoYoLAB 活動資料的分頁。

In [6]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1ybhLnih5Zwx3_JNecQF7vfPjY3XD1zAm4aP3zbLX-Is/edit?usp=sharing"
HOYOLAB_WORKSHEET_NAME = "RAG"
TIMEZONE_NOTE = "Asia/Taipei"

# Header for HoYoLAB API data, 根據使用者需求只保留特定欄位
WEBSITE_HEADER = [
    "event_id", "title", "url", "link", "created_at"
]

# HoYoLAB URLs
HOYOLAB_EVENTS_URL = "https://www.hoyolab.com/home/events"
# HoYoLAB API endpoint for event lists. This URL needs to be VERIFIED by inspecting
# network requests on hoyolab.com/home/events.
# Updated API URL for Genshin Impact events
HOYOLAB_API_LIST_URL = "https://bbs-api-os.hoyolab.com/community/community_contribution/wapi/event/list"

# HoYoLAB API 預設參數。您可以根據需求調整 gids (遊戲ID)、lang (語言)、region (地區)、limit (數量)。
# gids: 2 (原神), 6 (崩壞：星穹鐵道), 8 (絕區零)
HOYOLAB_API_PARAMS = {
    "gids": [2], # Changed to Genshin Impact (2)
    "lang": "en-us", # Changed to en-us based on the sample response
    "region": "us", # Changed to us based on the sample response
    "limit": 15, # Added limit as per user's sample URL
}

USER_AGENT = "Mozilla/5.0 (compatible; Colab Web Crawler)"

## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，確保連接到正確的 Google Sheet。

In [7]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：程式語言-測
🔗 https://docs.google.com/spreadsheets/d/1ybhLnih5Zwx3_JNecQF7vfPjY3XD1zAm4aP3zbLX-Is/edit?usp=sharing


In [8]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)

In [9]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")

In [10]:
def parse_hoyolab_event_data(item):
    """解析 HoYoLAB API 返回的單個活動資料。"""
    # The base URL for relative paths like /article/ID
    hoyolab_base_url = "https://www.hoyolab.com"

    event_id = str(item.get("id", ""))
    title = item.get("name", "")
    web_path = item.get("web_path", "")

    # Construct the full URL for what will be the 'link' (event page URL)
    if web_path and not (web_path.startswith("http://") or web_path.startswith("https://")):
        event_link = urljoin(hoyolab_base_url, web_path)
    else:
        event_link = web_path

    # The 'url' will now be the banner_url as requested
    event_banner_url = item.get("banner_url", "")

    # Convert Unix timestamp to ISO format for 'created_at'
    create_at_timestamp = item.get("create_at")
    if create_at_timestamp:
        # Convert timestamp to integer first
        created_at = datetime.fromtimestamp(int(create_at_timestamp)).isoformat(timespec="seconds")
    else:
        created_at = ""

    return {
        "event_id": event_id,
        "title": title,
        "url": event_banner_url, # Now banner_url is assigned to 'url'
        "link": event_link,      # Now web_path is assigned to 'link'
        "created_at": created_at,
    }

def crawl_hoyolab_with_api(gids, lang="zh-tw", region="tw", limit=20, max_pages=float('inf'), delay=0.5):
    """透過 HoYoLAB API 爬取活動列表。"""
    all_events = []
    current_offset = 0

    headers = {"User-Agent": USER_AGENT}

    print(f"🚀 正在從 HoYoLAB API 爬取活動 (gids={gids}, lang={lang}, region={region}, limit={limit})")

    for page_num in range(max_pages):
        params = {
            "gids": gids,
            "lang": lang,
            "region": region,
            "offset": current_offset,
            "limit": limit,
        }

        try:
            resp = requests.get(HOYOLAB_API_LIST_URL, params=params, headers=headers, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            if data.get("retcode") != 0:
                print(f"⚠️ API 返回錯誤：{data.get('message')}")
                break

            event_list = data.get("data", {}).get("list", [])
            if not event_list:
                print("ℹ️ 已無更多活動資料。")
                break

            for event_item in event_list:
                all_events.append(parse_hoyolab_event_data(event_item))

            print(f"📄 頁面 {page_num + 1} 已爬取 {len(event_list)} 筆活動。")

            current_offset = data.get("data", {}).get("next_offset")
            if not current_offset or data.get("data", {}).get("is_last"):
                break

            time.sleep(delay)

        except requests.exceptions.RequestException as e:
            print(f"❌ 請求 HoYoLAB API 失敗：{e}")
            break
        except Exception as e:
            print(f"❌ 處理 HoYoLAB API 響應失敗：{e}")
            break

    df = pd.DataFrame(all_events, columns=WEBSITE_HEADER)
    print(f"✅ 本次透過 HoYoLAB API 總共爬到 {len(df)} 筆活動資料")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 HoYoLAB 活動資料
3. 合併並用 `event_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功

In [11]:
# 執行 HoYoLAB API 爬蟲
import asyncio
import pandas as pd

try:
    # Prepare params for the crawler function, using HOYOLAB_API_PARAMS
    # Pass all items in HOYOLAB_API_PARAMS as keyword arguments
    # The max_pages can be set here if a specific number of pages is desired
    hoyolab_events_df_api = crawl_hoyolab_with_api(**HOYOLAB_API_PARAMS, max_pages=1) # Start with max_pages=1 for testing
    print("\n--- 爬取結果 --- ")
    display(hoyolab_events_df_api.head())

    # After successfully fetching with API, we can now ensure the worksheet and write the data
    # Create a worksheet for HoYoLAB events
    ws_hoyolab = ensure_worksheet(sh, HOYOLAB_WORKSHEET_NAME, WEBSITE_HEADER)
    print(f"✅ 已準備 worksheet：{ws_hoyolab.title}")

    # Read existing data from HoYoLAB worksheet
    old_hoyolab_df = read_sheet_df(ws_hoyolab, WEBSITE_HEADER)
    print(f"📌 Google Sheet 原本有 {len(old_hoyolab_df)} 筆 HoYoLAB 活動")

    # Combine new and old data, remove duplicates based on 'event_id'
    combined_hoyolab_df = pd.concat([old_hoyolab_df, hoyolab_events_df_api], ignore_index=True)
    combined_hoyolab_df = combined_hoyolab_df.drop_duplicates(subset=["event_id"], keep="last")
    combined_hoyolab_df = combined_hoyolab_df.sort_values(by="event_id", ascending=False) # 以 event_id 排序，確保新資料在前

    # Write back to Google Sheet
    written_count_hoyolab = write_sheet_df(ws_hoyolab, combined_hoyolab_df, WEBSITE_HEADER)
    print(f"✅ 已寫入 Google Sheet：{written_count_hoyolab} 筆 HoYoLAB 活動")

    # Verify write
    verify_hoyolab_df = read_sheet_df(ws_hoyolab, WEBSITE_HEADER)
    print(f"🔍 從 Google Sheet 重新讀回：{len(verify_hoyolab_df)} 筆 HoYoLAB 活動")

    if len(verify_hoyolab_df) == written_count_hoyolab:
        print("✅ HoYoLAB 活動寫入驗證成功")
    else:
        print("⚠️ HoYoLAB 活動寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")


except Exception as e:
    print(f"An error occurred during HoYoLAB API crawling and processing: {e}")
    print("請檢查 HoYoLAB API URL 和參數是否正確，或網路連線是否有問題。")

🚀 正在從 HoYoLAB API 爬取活動 (gids=[2], lang=en-us, region=us, limit=15)
📄 頁面 1 已爬取 17 筆活動。
✅ 本次透過 HoYoLAB API 總共爬到 17 筆活動資料

--- 爬取結果 --- 


,event_id,title,url,link,created_at
0,45409131,"The ""Time to Get Moving!"" web event is now ava...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45409131,2026-06-10T09:00:07
1,45389563,"Version ""Luna VII"" HoYoverse Top-Up Center Eve...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45389563,2026-06-09T04:09:05
2,45389493,Primogem Rewards: Participate in Lohen and Mav...,https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45389493,2026-06-09T03:59:06
3,45145556,"The ""Luna VII"" Genius Column Submission Event ...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45145556,2026-05-22T04:10:23
4,45098985,"The new limited-time web event ""Mage Nicole's ...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45098985,2026-05-20T04:00:07


✅ 已準備 worksheet：RAG
📌 Google Sheet 原本有 17 筆 HoYoLAB 活動
✅ 已寫入 Google Sheet：17 筆 HoYoLAB 活動
🔍 從 Google Sheet 重新讀回：17 筆 HoYoLAB 活動
✅ HoYoLAB 活動寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`網站活動資料 → Google Sheet → RAG`

In [12]:
# 從 Google Sheet 重新讀取 HoYoLAB 活動資料，作為 RAG 的唯一資料來源
ws_hoyolab = ensure_worksheet(sh, HOYOLAB_WORKSHEET_NAME, WEBSITE_HEADER)
rag_source_df = read_sheet_df(ws_hoyolab, WEBSITE_HEADER)

# 清掉沒有內容的文章 (由於現在沒有 'content' 欄位，這行會被跳過)
if "content" in rag_source_df.columns:
    rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

display(rag_source_df.head())

📚 可用於 RAG 的文章數：17


,event_id,title,url,link,created_at
0,45409131,"The ""Time to Get Moving!"" web event is now ava...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45409131,2026-06-10 9:00:07
1,45389563,"Version ""Luna VII"" HoYoverse Top-Up Center Eve...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45389563,2026-06-09 4:09:05
2,45389493,Primogem Rewards: Participate in Lohen and Mav...,https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45389493,2026-06-09 3:59:06
3,45145556,"The ""Luna VII"" Genius Column Submission Event ...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45145556,2026-05-22 4:10:23
4,45098985,"The new limited-time web event ""Mage Nicole's ...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45098985,2026-05-20 4:00:07


In [13]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")

def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        event_id = str(row.get("event_id", ""))
        title = str(row.get("title", ""))
        url = str(row.get("url", "")) # This is now the banner_url
        link = str(row.get("link", "")) # This is now the event page url
        created_at = str(row.get("created_at", ""))

        text = (f"活動標題：{title}\n"
                f"活動連結：{link}\n"
                f"活動圖片網址：{url}\n"
                f"創建時間：{created_at}") # Updated '活動網址' to '活動圖片網址'
        docs.append({
            "event_id": event_id,
            "title": title,
            "url": url,
            "link": link, # Added link to the document for easier retrieval
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成：17 篇文章，向量維度 384


---

## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`網站活動資料 → Google Sheet → RAG`

In [14]:
# 從 Google Sheet 重新讀取 HoYoLAB 活動資料，作為 RAG 的唯一資料來源
ws_hoyolab = ensure_worksheet(sh, HOYOLAB_WORKSHEET_NAME, WEBSITE_HEADER)
rag_source_df = read_sheet_df(ws_hoyolab, WEBSITE_HEADER)

# 清掉沒有內容的文章 (如果 content 欄位不存在，則跳過此篩選)
if "content" in rag_source_df.columns:
    rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()

📚 可用於 RAG 的文章數：17


,event_id,title,url,link,created_at
0,45409131,"The ""Time to Get Moving!"" web event is now ava...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45409131,2026-06-10 9:00:07
1,45389563,"Version ""Luna VII"" HoYoverse Top-Up Center Eve...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45389563,2026-06-09 4:09:05
2,45389493,Primogem Rewards: Participate in Lohen and Mav...,https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45389493,2026-06-09 3:59:06
3,45145556,"The ""Luna VII"" Genius Column Submission Event ...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45145556,2026-05-22 4:10:23
4,45098985,"The new limited-time web event ""Mage Nicole's ...",https://upload-os-bbs.hoyolab.com/upload/2026/...,https://www.hoyolab.com/article/45098985,2026-05-20 4:00:07


In [15]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")

def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        event_id = str(row.get("event_id", ""))
        title = str(row.get("title", ""))
        url = str(row.get("url", ""))
        link = str(row.get("link", ""))
        created_at = str(row.get("created_at", ""))
        # 注意：由於 Google Sheet 不再包含 'content' 欄位，這裡的 RAG 文件將無法使用詳細內容
        # RAG 效果可能因此受限

        text = (f"活動標題：{title}\n"
                f"活動連結：{link}\n"
                f"活動網址：{url}\n"
                f"創建時間：{created_at}")
        docs.append({
            "event_id": event_id,
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成：17 篇文章，向量維度 384


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。

In [16]:
api_key = userdata.get("GE2.5-F-L")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-3-flash-preview


In [17]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關網站活動資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據HOYOLAB網站Genshin Impact活動資料回答問題的助教。
請只根據【活動資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【活動資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


In [18]:
!pip -q install gradio

## 8. 建立 Gradio 介面

In [19]:
import gradio as gr

def gradio_query_rag(question):
    """Gradio 介面調用 RAG 模型的函數。"""
    if not question.strip():
        return "請輸入您的問題。"
    return query_rag(question, k=3)

# 假設 rag_source_df 已經從 Google Sheet 載入
# 如果 rag_source_df 在 Gradio 執行時未定義，Gradio 介面將無法正確顯示表格
# 因此確保 rag_source_df 已經執行過 cell HtGQxrShTPyH

# 只選擇需要的欄位來顯示
display_df = rag_source_df[["event_id", "title", "url", "link", "created_at"]]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# HOYOLAB原神版活動資訊檢索RAG")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 提問區")
            question_input = gr.Textbox(lines=5, label="輸入問題：", placeholder="例如：最近有什麼原神活動？或是：45409131這個活動的詳情是什麼？")
            submit_button = gr.Button("確認提問")
            answer_output = gr.Textbox(label="RAG 回答：", lines=10)

        with gr.Column(scale=2):
            gr.Markdown("### 活動資訊總覽 (來自 Google Sheet)")
            gr.DataFrame(value=display_df, headers=list(display_df.columns), label="HoYoLAB 活動資料", wrap=False) # Added wrap=False here

    submit_button.click(
        fn=gradio_query_rag,
        inputs=question_input,
        outputs=answer_output
    )

demo.launch(debug=True)

/tmp/ipykernel_47398/1402242756.py:16: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://74b00d0541a1666cb4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5146.79ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 912.84ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 53122.81ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 736.35ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 868.76ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1269.13ms


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://74b00d0541a1666cb4.gradio.live


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。


## 什麼是 RAG (Retrieval Augmented Generation)？

RAG (Retrieval Augmented Generation) 是一種結合了檢索 (Retrieval) 和生成 (Generation) 技術的自然語言處理 (NLP) 框架。

**核心概念：**

傳統的生成式 AI 模型（如大型語言模型 LLM）在生成內容時，主要依賴其訓練資料中學到的知識。然而，這些模型有時會面臨以下問題：

1.  **知識陳舊：** 訓練資料通常不是最新的，導致模型無法回答關於近期事件的問題。
2.  **事實錯誤 (Hallucination)：** 模型可能會「編造」事實上不存在的資訊。
3.  **缺乏領域特定知識：** 對於特定領域（如醫療、法律或企業內部資料）的專業問題，模型可能缺乏足夠的訓練資料來給出準確的答案。

RAG 透過在生成答案之前，從外部知識庫（例如文件、資料庫或網頁）中檢索相關資訊，來解決這些問題。這個過程可以分為兩個主要階段：

1.  **檢索 (Retrieval)：**
    *   當使用者提出問題時，RAG 系統首先會分析問題，並將其轉換成一個查詢 (query)。
    *   接著，系統會利用這個查詢在一個大型的外部知識庫中搜尋最相關的文件或文本片段。
    *   通常會使用向量搜尋技術（例如 FAISS 索引）來找到與查詢語義最相似的內容。

2.  **生成 (Generation)：**
    *   檢索到的相關資訊（上下文，Context）會與原始問題一起被送入生成式 AI 模型 (例如 Gemini)。
    *   AI 模型會利用這些額外的上下文資訊，生成一個更準確、更具相關性且基於事實的答案。

**RAG 的優點：**

*   **提高準確性：** 減少模型「編造」資訊的可能性，使回答更可靠。
*   **提供最新資訊：** 知識庫可以定期更新，確保模型能夠回答關於最新事件的問題。
*   **引入領域特定知識：** 可以為特定應用場景建立專屬的知識庫，使模型在專業領域表現更出色。
*   **可追溯性：** 由於答案是基於檢索到的特定文件，使用者可以追溯資訊的來源。
*   **降低模型再訓練成本：** 無需每次更新知識就重新訓練整個大型模型，只需更新檢索的知識庫即可。

**總之，RAG 讓生成式 AI 模型不僅能「說」，還能「查」，大大提升了回答的品質和可靠性。**